# Local execution

So far, every circuit has been run with `infrastructure="local"`. This notebook explains what that backend does, how to spread shots across several runs with `n_qpus`, and how to fix randomness with `seed`. [`03b_scaling_cunqa_qmio.ipynb`](03b_scaling_cunqa_qmio.ipynb) covers the infrastructures that need a cluster: CUNQA and QMIO.

## What is behind "local"

`infrastructure="local"` runs the circuit by default with Qiskit's Aer simulator, on the computer itself. `result.backend` confirms it:

In [ ]:
import polypus

bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()
result = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local")
print(result.backend)

## The native backend

Besides `"aer"`, Polypus has its own simulation engine written in Rust, selectable with the `backend` argument:

In [ ]:
native_result = polypus.run_quantum_circuit(
    bell, shots=1000, infrastructure="local", backend="polypus"
)
print(native_result.backend, native_result.counts)

The result is the same Bell state, only the engine computing it changes. The native backend also supports `fusion`, which combines consecutive gates before simulating to run faster: it is on by default, and `fusion=False` forces a gate-by-gate run. Passing `fusion=True` on `"aer"`, which does not know how to fuse, raises an error instead of silently ignoring it:

In [ ]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="local", backend="aer", fusion=True
    )
except ValueError as e:
    print(e)

The performance differences between both backends are documented, with reproducible benchmarks, in the main README.

## Spreading shots with `n_qpus`

`n_qpus` spreads the shots across several instances in parallel, to reduce wall-clock time. With `n_qpus=1`, the default, the result comes wrapped in a list with a single element, as seen in [`01_first_circuit.ipynb`](01_first_circuit.ipynb):

In [ ]:
r1 = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", n_qpus=1)
print(r1.counts)

With `n_qpus` greater than 1, Polypus merges the results of all instances into a single dictionary, with no list:

In [ ]:
r4 = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", n_qpus=4)
print(r4.counts)

This change of shape is a real source of bugs: code that assumes `result.counts[0]` stops working as soon as `n_qpus` goes from 1 to a higher value, and raises a `KeyError`, because a dictionary of bitstrings has no key `0`:

In [ ]:
try:
    r4.counts[0]
except KeyError as e:
    print("KeyError:", e)

That is why it helps to check the type before accessing it, or simply to treat `result.counts` as whatever it actually is in each case: a list with one element, or an already-merged dictionary.

## Reproducibility with `seed`

By default, every run uses a different random seed, taken from the operating system. Passing `seed` fixes that randomness, and the result is identical across runs:

In [ ]:
a = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", seed=42)
b = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", seed=42)
print(a.counts == b.counts)

Without passing `seed`, `result.seed` reports the seed that was used, to reproduce that same run later:

In [ ]:
unseeded_result = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local")
print(unseeded_result.seed)

## Other result fields

Besides `counts`, `seed`, and `backend`, `result.id` stores a unique identifier for the run, and `result.infrastructure` repeats the `infrastructure` value used in the call. They are useful for logging and finding a specific run later.

## Summary

This notebook explained what `infrastructure="local"` does, the choice between the `"aer"` backend and the native `"polypus"` one with `fusion`, how `n_qpus` changes both the shot distribution and the shape of the result, with the change from list to merged dictionary as a real source of bugs, and how `seed` gives reproducibility.

The next section covers the infrastructures that need a cluster: CUNQA and QMIO.

## Next step

Continue with: [`03b_scaling_cunqa_qmio.ipynb`](03b_scaling_cunqa_qmio.ipynb).